# Extract Central Banks From Wikipedia V2

Esta versión corrige los problemas principales del notebook original:

1. Usa un solo flujo de extracción.
2. Conserva el `wikipedia_bank_url` de cada banco central.
3. Extrae filas crudas de tablas e infoboxes relacionadas con governors, presidents, chairs o key people.
4. Cuando una tabla viene con claves numéricas o headers repetidos dentro del cuerpo, guarda:
   - `row_data`: la fila tal como salió de la tabla
   - `resolved_row_data`: la fila relabelled usando el header detectado, si aplica
   - `row_kind`: `header` o `data`
5. Exporta también `request_errors_df` para no perder trazabilidad de timeouts o páginas fallidas.

## Salidas principales

### `central_banks_df`
- `country`
- `central_bank`
- `wikipedia_bank_url`

### `central_bank_governors_df`
- `country`
- `central_bank`
- `wikipedia_bank_url`
- `source_type`
- `source_label`
- `row_kind`
- `row_data`
- `resolved_row_data`

### `request_errors_df`
- filas con errores de request al visitar páginas de bancos centrales


In [ ]:
import json
import re
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CentralBankNotebook/2.0)"}
WIKI_BANKS_URL = "https://en.wikipedia.org/wiki/List_of_central_banks"
ROLE_PATTERNS = ["governor", "governors", "president", "presidents", "chair", "chairs", "key people"]
HEADER_HINTS = {
    "name", "governor", "president", "chair", "chairman", "term", "period",
    "from", "until", "start", "end", "took office", "left office", "notes",
    "comments", "no", "#", "entered office", "exited office", "ref"
}


def clean_text(value):
    value = str(value or "")
    value = re.sub(r"\[[^\]]*\]", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip(" -–—,;")


def flatten_columns(columns):
    if not isinstance(columns, pd.MultiIndex):
        return [clean_text(col) for col in columns]
    flattened = []
    for col in columns:
        parts = [clean_text(x) for x in col if clean_text(x) and str(x) != "nan"]
        flattened.append(" | ".join(parts))
    return flattened


def table_context_text(table):
    pieces = []
    caption = table.find("caption")
    if caption:
        pieces.append(caption.get_text(" ", strip=True))
    rows = table.select("tr")
    if rows:
        pieces.append(" ".join(cell.get_text(" ", strip=True) for cell in rows[0].find_all(["th", "td"])))
    prev = table.find_previous(["h2", "h3", "h4"])
    if prev:
        pieces.append(prev.get_text(" ", strip=True))
    return clean_text(" | ".join(piece for piece in pieces if piece))


def sanitize_table_html(table):
    html = str(table)
    html = re.sub(r'colspan="(\d+);"', r'colspan="\1"', html)
    html = re.sub(r'rowspan="(\d+);"', r'rowspan="\1"', html)
    return html


def parse_html_table(table):
    return pd.read_html(StringIO(sanitize_table_html(table)))[0]


def normalize_label(label):
    label = clean_text(label).lower()
    label = label.replace("№", "no").replace("#", "no")
    return label


def is_numeric_like(label):
    label = clean_text(label)
    return label == "" or label.isdigit()


def looks_like_header_row(row_data):
    values = {normalize_label(v) for v in row_data.values() if clean_text(v)}
    return len(values & HEADER_HINTS) >= 2


def normalize_header_map(row_data):
    return {clean_text(k): clean_text(v) for k, v in row_data.items() if clean_text(k) and clean_text(v)}


def relabel_row_data(row_data, header_map):
    relabeled = {}
    for key, value in row_data.items():
        cleaned_key = clean_text(key)
        cleaned_value = clean_text(value)
        if not cleaned_value:
            continue
        new_key = header_map.get(cleaned_key, cleaned_key)
        relabeled[new_key] = cleaned_value
    return relabeled


def build_row_payloads(parsed_table):
    parsed_table = parsed_table.copy()
    parsed_table.columns = flatten_columns(parsed_table.columns)

    payloads = []
    header_map = {}

    for _, row in parsed_table.iterrows():
        row_data = {
            clean_text(col): clean_text(row.get(col, ""))
            for col in parsed_table.columns
            if clean_text(row.get(col, ""))
        }
        if not row_data:
            continue

        resolved_row_data = dict(row_data)
        row_kind = "data"

        if all(is_numeric_like(col) for col in row_data.keys()) and looks_like_header_row(row_data):
            header_map = normalize_header_map(row_data)
            resolved_row_data = dict(header_map)
            row_kind = "header"
        elif header_map and any(is_numeric_like(col) for col in row_data.keys()):
            resolved_row_data = relabel_row_data(row_data, header_map)

        payloads.append({
            "row_kind": row_kind,
            "row_data": row_data,
            "resolved_row_data": resolved_row_data,
        })

    return payloads


def extract_table_records(parsed_table, bank_row, context_text):
    records = []
    for payload in build_row_payloads(parsed_table):
        records.append({
            "country": clean_text(bank_row.country),
            "central_bank": clean_text(bank_row.central_bank),
            "wikipedia_bank_url": clean_text(bank_row.wikipedia_bank_url),
            "source_type": "wikitable",
            "source_label": context_text,
            "row_kind": payload["row_kind"],
            "row_data": payload["row_data"],
            "resolved_row_data": payload["resolved_row_data"],
        })
    return records


def extract_infobox_records(bank_row, soup):
    records = []
    infobox = soup.select_one("table.infobox")
    if not infobox:
        return records

    for tr in infobox.select("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if not th or not td:
            continue

        label = clean_text(th.get_text(" ", strip=True))
        value = clean_text(td.get_text(" ", strip=True))
        if not label or not value:
            continue

        if any(pattern in label.lower() for pattern in ROLE_PATTERNS):
            row_data = {label: value}
            records.append({
                "country": clean_text(bank_row.country),
                "central_bank": clean_text(bank_row.central_bank),
                "wikipedia_bank_url": clean_text(bank_row.wikipedia_bank_url),
                "source_type": "infobox",
                "source_label": label,
                "row_kind": "data",
                "row_data": row_data,
                "resolved_row_data": dict(row_data),
            })
    return records


def extract_governor_rows(bank_row):
    url = bank_row.wikipedia_bank_url
    if not url:
        return []

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        return [{
            "country": clean_text(bank_row.country),
            "central_bank": clean_text(bank_row.central_bank),
            "wikipedia_bank_url": clean_text(url),
            "source_type": "request_error",
            "source_label": type(exc).__name__,
            "row_kind": "error",
            "row_data": {"error": str(exc)},
            "resolved_row_data": {"error": str(exc)},
        }]

    soup = BeautifulSoup(response.text, "html.parser")
    records = []

    for table in soup.select("table.wikitable"):
        context_text = table_context_text(table)
        if not any(pattern in context_text.lower() for pattern in ROLE_PATTERNS):
            continue
        try:
            parsed_table = parse_html_table(table)
        except ValueError:
            continue
        records.extend(extract_table_records(parsed_table, bank_row, context_text))

    if records:
        return records
    return extract_infobox_records(bank_row, soup)


# 1. Central banks list + wikipedia links
response = requests.get(WIKI_BANKS_URL, headers=HEADERS, timeout=30)
response.raise_for_status()

wiki_tables = pd.read_html(StringIO(response.text))
soup = BeautifulSoup(response.text, "html.parser")
html_tables = soup.select("table.wikitable")

if len(wiki_tables) < 2 or len(html_tables) < 2:
    raise ValueError("Expected at least two tables on the Wikipedia page")

alphabetical_table_raw = wiki_tables[0].copy()
major_table_raw = wiki_tables[1].copy()
alphabetical_html_table = html_tables[0]

alphabetical_records = []
for row in alphabetical_html_table.select("tr")[1:]:
    cols = row.find_all(["th", "td"])
    if len(cols) < 3:
        continue

    link = cols[2].find("a", href=True)
    wikipedia_bank_url = ""
    if link and link["href"].startswith("/wiki/"):
        wikipedia_bank_url = "https://en.wikipedia.org" + link["href"]

    alphabetical_records.append({
        "country": cols[0].get_text(" ", strip=True),
        "central_bank": cols[2].get_text(" ", strip=True),
        "wikipedia_bank_url": wikipedia_bank_url,
    })

alphabetical_banks_df = pd.DataFrame(alphabetical_records)
major_banks_df = major_table_raw.rename(
    columns={
        major_table_raw.columns[0]: "country",
        major_table_raw.columns[1]: "central_bank",
    }
)[["country", "central_bank"]].copy()
major_banks_df["wikipedia_bank_url"] = ""

for frame in (alphabetical_banks_df, major_banks_df):
    frame["country"] = frame["country"].astype(str).str.strip()
    frame["central_bank"] = frame["central_bank"].astype(str).str.strip()
    frame["wikipedia_bank_url"] = frame["wikipedia_bank_url"].astype(str).str.strip()

central_banks_df = pd.concat([alphabetical_banks_df, major_banks_df], ignore_index=True)
central_banks_df = central_banks_df.drop_duplicates(subset=["country", "central_bank"]).reset_index(drop=True)

print("central_banks_df", central_banks_df.shape)
display(central_banks_df.head(10))

# 2. Raw extraction from each bank page
rows = []
for i, bank_row in enumerate(central_banks_df.itertuples(index=False), start=1):
    if i % 25 == 0:
        print(f"Processed {i}/{len(central_banks_df)} banks")
    rows.extend(extract_governor_rows(bank_row))

central_bank_governors_df = pd.DataFrame(rows)
request_errors_df = central_bank_governors_df[central_bank_governors_df["source_type"] == "request_error"].copy()

print("central_bank_governors_df", central_bank_governors_df.shape)
print("request_errors_df", request_errors_df.shape)
display(central_bank_governors_df.head(20))

# 3. Export
central_banks_df.to_csv("data/central_banks_v2.csv", index=False, sep=";")
central_bank_governors_df.to_csv("data/central_bank_governors_v2.csv", index=False, sep=";")
request_errors_df.to_csv("data/central_bank_governors_request_errors_v2.csv", index=False, sep=";")
